# KulitKu — Eksperimen CNN beku × classifier ML
## Satu notebook, satu konfigurasi, implementasi dari src

Urutan: **Setup → EDA → Cleaning → EDA lagi → Split → Fitur CNN → CV classifier → Evaluasi validation → Benchmark/pilih → Test final → ONNX**.

Tidak ada fungsi penelitian yang disalin ke notebook. Semua tahap mengimpor `kulitku_ml` dari `src`. `experiment.json` dibaca dan digunakan langsung; run_config.json di Drive adalah salinan aktual, bukan konfigurasi kedua untuk diedit. Grafik dan tabel langsung disimpan saat dibuat.

**PENTING tentang dataset augmentasi:** default `exploratory_hash` bisa dipakai mencoba pipeline tanpa identitas gambar asal, tetapi hanya menahan duplikat byte persis. Variasi crop/rotasi/flip atau foto pasien sama belum tentu berada dalam kelompok yang sama. Semua hasil diberi status EKSPLORATORI dan tidak membuktikan generalisasi klinis. Untuk eksperimen final, gunakan `verified_groups` dan CSV lineage berisi bukti; contoh CSV dan petunjuk ada di configs/ dan README.

Tidak ada data, weights medis atau hasil akurasi penelitian yang disertakan. Pretrained weights diunduh saat ekstraksi pertama. GPU T4 membantu ekstraksi; classifier sklearn berjalan di CPU. Colab gratis mungkin membutuhkan beberapa sesi.


## 1. Persiapkan kode di Colab
**Cara ZIP (tanpa harus membuat GitHub dahulu):** ekstrak ZIP di laptop, upload notebook ini ke Colab, lalu jalankan sel berikut. Saat diminta, upload ZIP paket utuh ke runtime Colab (bukan dataset). Kode mengekstrak folder KulitKu di /content.

**Cara GitHub:** upload isi folder KulitKu ke repository-mu, isi REPO_URL dan REPO_REF. Untuk repo private gunakan ZIP agar tidak perlu menaruh token pada notebook. Sel tidak menimpa checkout yang sudah ada; kode lokal harus sama dengan ref yang dimaksud.

Tidak mengimpor ML library sebelum instalasi agar mengurangi modul versi lama di memori.


In [ ]:
from pathlib import Path
import os,sys,subprocess,zipfile,json
REPO_URL = ''   # opsional URL HTTPS github.com tanpa token; kosong = upload ZIP
REPO_REF = 'main'
CODE_ROOT = Path('/content/KulitKu/machine-learning')
if not (CODE_ROOT/'pyproject.toml').exists():
    if REPO_URL:
        from urllib.parse import urlparse
        parsed=urlparse(REPO_URL)
        if parsed.scheme!='https' or parsed.hostname!='github.com' or parsed.username or parsed.password:
            raise ValueError('Gunakan URL GitHub HTTPS tanpa kredensial.')
        target=Path('/content/KulitKu')
        if target.exists():raise FileExistsError('Direktori kode sudah ada, periksa isinya tanpa menghapus pekerjaan.')
        subprocess.run(['git','clone','--branch',REPO_REF,REPO_URL,str(target)],check=True)
    else:
        from google.colab import files
        print('Upload ZIP paket KulitKu lengkap, bukan ZIP dataset.')
        uploaded=files.upload()
        candidates=[Path(n) for n in uploaded if n.lower().endswith('.zip')]
        if len(candidates)!=1:raise ValueError('Upload tepat satu ZIP paket.')
        with zipfile.ZipFile(candidates[0]) as archive:
            dest=Path('/content').resolve()
            for item in archive.infolist():
                resolved=(dest/item.filename).resolve()
                if not resolved.is_relative_to(dest):raise ValueError('Path ZIP tidak aman.')
                if ((item.external_attr>>16)&0o170000)==0o120000:raise ValueError('Symlink ZIP tidak diizinkan.')
            archive.extractall(dest)
if not (CODE_ROOT/'src/kulitku_ml/config.py').exists():raise FileNotFoundError('Struktur kode tidak sesuai; harus /content/KulitKu/machine-learning.')
print('Kode:',CODE_ROOT)


## 2. Instal dependensi
Pasangan torch/torchvision CUDA bawaan Colab digunakan bila memenuhi versi yang dibutuhkan. pandas/Pillow tidak dipaksa ke versi yang bertentangan dengan Colab. requirements memakai pin untuk komponen model/ONNX dan rentang untuk library bawaan; versi aktual direkam di environment.json.

Jika Colab meminta restart, restart session lalu jalankan bagian 1, lewati instalasi yang sudah selesai dengan INSTALL_DEPENDENCIES=False, dan lanjut bagian 3. Jangan mencampur versi dalam satu run.


In [ ]:
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(CODE_ROOT/'requirements.txt')],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-deps','-e',str(CODE_ROOT)],check=True)
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
print(check.stdout or check.stderr)
if check.returncode:
    raise RuntimeError('Konflik paket terdeteksi. Selesaikan sebelum menjalankan eksperimen; jangan menonaktifkan pemeriksaan.')


## 3. Mount Drive dan baca satu konfigurasi
Sesuaikan path dalam `configs/experiment.json` sebelum menjalankan, atau isi dua override path di bawah. Override dicatat di salinan run_config.json. Pengaturan model/grid/fold tetap dibaca dari JSON, bukan digandakan dalam sel.

RUN_ID kosong membuat eksperimen baru. Untuk melanjutkan sesi terputus, isi RUN_ID yang tercetak/disimpan; kode/config/versi paket harus sama dan dataset tidak berubah. Cache tersedia per CNN dan per kandidat/fold classifier yang selesai. Batch fitur CNN yang belum selesai harus diulang.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Explicit path to src ensures the notebook uses this checkout after restart.
sys.path.insert(0,str(CODE_ROOT/'src'))
import torch,pandas as pd
from IPython.display import display
from kulitku_ml.config import load_config,Run
DATASET_ROOT_OVERRIDE = ''  # contoh /content/drive/MyDrive/datasets/original
OUTPUT_ROOT_OVERRIDE = ''
RUN_ID = ''
cfg=load_config(CODE_ROOT/'configs/experiment.json')
if DATASET_ROOT_OVERRIDE:cfg['paths']['dataset_root']=DATASET_ROOT_OVERRIDE
if OUTPUT_ROOT_OVERRIDE:cfg['paths']['output_root']=OUTPUT_ROOT_OVERRIDE
# Before CUDA operations; strict deterministic errors should be resolved, not silently ignored.
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
run=Run(cfg,RUN_ID or None)
print('RUN_ID:',run.id,'| Data:',run.root,'| Hasil:',run.out)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Mode split:',cfg['split']['mode'])
if run.exploratory:print('HASIL EKSPLORATORI: hubungan augmentasi/pasien belum terverifikasi.')
# Snapshot versions only; omit direct URL/local lines that could contain credentials.
freeze=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout
safe_lines=[line for line in freeze.splitlines() if '://' not in line and ' @ ' not in line and not line.startswith('-e ')]
(run.out/'requirements_snapshot.txt').write_text('\n'.join(safe_lines)+'\n')


## 4. Inventaris dan EDA awal
Sel ini membaca setiap file untuk decode, format dan resolusi; hash belum dihitung. Jika data masih di Drive, I/O bisa lambat. Boleh arahkan dataset_root ke salinan lokal Colab dari dataset yang sama; pastikan salinan memang lengkap dan jangan mengubah sumber dalam satu run. Mengganti path mengharuskan run baru.

Tabel, grafik resolusi/format/kelas, galeri dan indikator visual sampel langsung disimpan ke Drive. Indikator brightness/Laplacian tidak menjadi alasan eksklusi otomatis.


In [ ]:
from kulitku_ml.data import inventory
from kulitku_ml.eda import eda,compare
before=inventory(run)
display(eda(run,before,'before'))


## 5. Cleaning dan EDA setelah cleaning
Cleaning teknis tidak menghapus sumber. Review manual opsional memakai tabel/manual_review_template.csv. Untuk menerapkannya, isi path cleaning.manual_review_csv pada konfigurasi lalu buat run baru, agar keputusan lama tidak tercampur.

Output cleaned.csv adalah daftar kandidat; tidak menyalin seluruh foto dan belum menjamin bebas duplikat. Mode grayscale/alpha yang bisa dikonversi RGB tetap dipertahankan. Minimum 128px default hanya flag. Transformasi input CNN belum dilakukan; diterapkan saat ekstraksi.


In [ ]:
from kulitku_ml.cleaning import clean
after=clean(run,before)
display(compare(run,before,after))
display(eda(run,after,'after'))
print('Kandidat setelah cleaning:',len(after))


## 6. Split development/test dan lima fold
Hash byte dilakukan pada tahap ini untuk exact duplicate, bukan pada EDA. Konflik label duplikat harus diselesaikan melalui review; proses berhenti dengan tabel yang menunjukkan konflik, tidak menebak label yang benar.

Default: holdout 1 dari 5 kelompok (~20% test) dan lima fold development. `verified_groups` memakai kelompok asal ber-bukti, validation/test original terverifikasi, turunannya hanya training kelompok terkait. Default `exploratory_hash` hanya menahan exact duplicate dan tetap berisiko kebocoran augmentasi. Tidak ada klaim split antar pasien karena patient_id tidak tersedia.


In [ ]:
from kulitku_ml.splitting import split
splits=split(run,after)
display(run.read('split_summary.csv'))


## 7. Ekstraksi tiga CNN beku
Seluruh bobot tidak di-update; mode eval. CNN menghasilkan vektor fitur sebelum kepala klasifikasi. Ini menghemat komputasi dan membuat perbandingan fitur lebih jelas. Classifier yang dilatih menggunakan label KulitKu, bukan CNN-nya.

Pretrained weights timm diunduh pertama kali. Preprocessing spesifik weights disimpan dan digunakan ulang. Fitur seluruh gambar dapat di-cache karena ekstraktor beku/deterministik; tidak ada scaler atau komponen terlatih yang di-fit menggunakan test. Cache tidak digunakan jika urutan split berubah.

Jalankan satu CNN dahulu jika perlu; semua nama diambil dari JSON. TRAIN_ONLY_CNN/CLASSIFIER pada tahap berikut hanya filter eksekusi, bukan perubahan protokol/grid.


In [ ]:
from kulitku_ml.features import extract_features
EXTRACT_ONLY = ''  # kosong=semua; atau mobilenetv4 / resnet50 / convnext_tiny
for cnn in cfg['extractors']:
    if EXTRACT_ONLY and cnn!=EXTRACT_ONLY:continue
    x=extract_features(run,splits,cnn)
    print(cnn,x.shape)


## 8. Training tiga classifier dan 5-fold CV
Setiap CNN × LR/SVM/RF menghasilkan sembilan kombinasi. Scaler LR/SVM di-fit hanya pada training setiap fold. RF tanpa scaler. Grid default 26 kandidat per CNN × 5 fold = 390 fit pencarian, ditambah refit dan pembuatan prediksi validation terpilih. Bisa lama pada Colab gratis.

Model terbaik tiap kombinasi di-refit pada seluruh development untuk benchmarking; test tidak dilihat. Progress disimpan setiap kandidat/fold sehingga run yang sama dapat dilanjutkan. Jangan load joblib dari sumber tak dipercaya.


In [ ]:
from kulitku_ml.training import train_classifier
import numpy as np
TRAIN_ONLY_CNN = ''
TRAIN_ONLY_CLASSIFIER = ''
for cnn in cfg['extractors']:
    if TRAIN_ONLY_CNN and cnn!=TRAIN_ONLY_CNN:continue
    x=np.load(run.out/'features'/cnn/'features.npy',mmap_mode='r')
    for name in cfg['classifiers']:
        if TRAIN_ONLY_CLASSIFIER and name!=TRAIN_ONLY_CLASSIFIER:continue
        result=train_classifier(run,splits,x,cnn,name)
        print(result['combo_id'],result['mean_macro_f1'])


## 9. Evaluasi validation dan visualisasi sembilan kombinasi
Confusion matrix jumlah/normalisasi, precision-recall-F1 per kelas, galeri salah, heatmap 3×3, mean±SD dan skor setiap fold. Skor validation ini sudah dipakai tuning, sehingga bukan evaluasi akhir independen.


In [ ]:
from kulitku_ml.evaluation import validation_report,cv_overview
for cnn in cfg['extractors']:
    for name in cfg['classifiers']:
        validation_report(run,cnn,name)
display(cv_overview(run))


## 10. Benchmark CPU dan kunci pilihan
Semua kandidat diukur dengan runtime PyTorch CPU + sklearn, batch1, jumlah thread yang sama dan sampel development yang sama. Timer termasuk decode/preprocessing/CNN/scaler/classifier, tidak termasuk jaringan atau pemuatan model. RF prediction jobs=1 untuk benchmark. Model refit development telah tersedia dari tahap training; tidak perlu fit ulang yang identik.

Aturan: macro F1 rata-rata maksimal 0.01 di bawah terbaik, lalu median latency tercepat, ukuran lebih kecil dan combo_id sebagai tie-break deterministik. Kunci disimpan sebelum membuka test. Benchmark ini membandingkan runtime native; ONNX bisa memiliki urutan kecepatan berbeda, dan tidak boleh diklaim kandidat tercepat ONNX tanpa profiling semua kandidat lagi.


In [ ]:
from kulitku_ml.benchmarking import benchmark
display(benchmark(run,splits))
print((run.out/'selection.json').read_text())


## 11. Test final kombinasi terpilih
Hanya pilihan yang sudah dikunci dievaluasi. Tidak tuning ulang atau mengganti pemenang berdasarkan test. Tersedia classification report, confusion matrix, metrik kelas, galeri salah, CI bootstrap kelompok dan grafik CV vs test. Bootstrap group tidak menjamin independensi antar pasien.


In [ ]:
from kulitku_ml.evaluation import final_test
selected=json.loads((run.out/'selection.json').read_text())
display(final_test(run,splits,selected['cnn'],selected['classifier']))


## 12. Ekspor ONNX dan parity check
Ekspor CNN dan pipeline scaler/classifier. Sklearn converter mendukung classifier yang digunakan, tetapi numerik harus diverifikasi. Parity memakai gambar development, membandingkan fitur dan label native vs ONNX. Jika gagal, proses berhenti dan tidak membuat release siap inferensi.

labels.json dan preprocessing.json benar-benar dikonsumsi oleh Predictor pada bagian 13. Tidak ada persentase kepastian medis yang diklaim. Preprocessing masih di luar grafik ONNX; kedua model ONNX menerima tensor/fitur, bukan JPEG.


In [ ]:
from kulitku_ml.export import export_selected
release_dir=export_selected(run,splits)
print('Paket:',release_dir)


## 13. Uji paket inferensi hasil ekspor
Predictor membaca release, checksum, label, dan konfigurasi preprocessing, lalu menjalankan kedua file ONNX. Ini modul inferensi untuk calon integrasi FastAPI, bukan layanan FastAPI/Flutter/Supabase lengkap. Waktu satu panggilan ini bukan benchmark aplikasi yang mencakup upload/jaringan.


In [ ]:
from kulitku_ml.inference import Predictor
predictor=Predictor(release_dir,threads=cfg['benchmark']['threads'])
example=splits[splits.partition.eq('development')].iloc[0]
print(predictor.predict(run.root/example.relative_path))
print('Selesai. Semua hasil:',run.out)


## File yang benar-benar dipakai
| Bagian | Pemanggil |
|---|---|
| configs/experiment.json | load_config di bagian 3 |
| src/kulitku_ml/config.py | konfigurasi, folder otomatis, penyimpanan output |
| data.py dan eda.py | bagian 4–5 |
| cleaning.py | bagian 5 |
| splitting.py | bagian 6 dan indeks CV pada training |
| preprocessing.py dan features.py | bagian 7, benchmarking, export, inferensi |
| training.py dan evaluation.py | bagian 8–9 dan 11 |
| benchmarking.py | bagian 10 |
| export.py dan inference.py | bagian 12–13 |

Folder Drive otomatis: grafik/, tabel/, features/<cnn>/, models/<cnn>/<classifier>/, onnx/<pilihan>/. CSV cleaning dan split tetap diperlukan untuk keterlacakan, tetapi tidak ada folder manifests terpisah. Notebook tidak membuat salinan dataset hasil cleaning; kode membaca daftar gambar yang dipertahankan.

Keluaran adalah riset skrining awal, bukan diagnosis atau validasi klinis. Semua keterbatasan lineage/dataset tetap berlaku setelah export.
